# MeanFlow — 10-class Imagenette training (bonus tier)

Trains the unconditional MeanFlow model on the full 10-class Imagenette train split (~9.5k images, 32×32 pixel space) and watches one-step samples evolve.

**Goal (from the assignment):** one-step samples on the training set look reasonable — *recognizable structure, not noise*. Full held-out quality is not required.

**Before running:** Runtime → Change runtime type → GPU (L4 preferred, T4 works).

Expect the main run to take several hours. Colab VMs are ephemeral: run the **Save to Drive** cell periodically, and use the **Resume** cell after a disconnect.

In [ ]:
!git clone -b imagenette10 https://github.com/nonplayercharacter3/MeanFlow-one-step-pixel-generation.git meanflow
%cd meanflow
!nvidia-smi -L

## Get the dataset

Imagenette (160px edition) from fastai's S3 bucket, extracted to where `--data-dir` expects it.

In [ ]:
!wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz -O data/imagenette2-160.tgz
!tar -xzf data/imagenette2-160.tgz -C data/
!find data/imagenette2-160/train -name '*.JPEG' | wc -l

## Smoke test (~2 min)

A short run with a small model: verifies the JVP check, the data pipeline, and that loss moves before committing hours to the real run.

In [ ]:
!python train.py --data-dir data/imagenette2-160/train \
  --steps 200 --batch-size 64 --hidden-channels 64 --num-blocks 1 \
  --eval-every 20 --sample-every 100 --checkpoint-every 200 \
  --num-eval-noises 16 --num-workers 2 --output-dir outputs/imagenette10_smoke

## Main run

The paper-flavored recipe: logit-normal time sampling, 50% plain flow-matching fraction, adaptive loss weighting p=1 (the paper's ImageNet setting), cosine LR over the full 60k steps, EMA 0.999.

Structure typically emerges gradually: colored blobs by ~5–10k steps, scene-like layout later. Watch the `sample_step_*.png` grids (visualization cell below) rather than the raw loss.

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train.py --data-dir data/imagenette2-160/train \
  --steps 60000 --batch-size 64 --lr 2e-4 --lr-decay-steps 60000 \
  --time-sampling logit_normal --equal-time-probability 0.5 --endpoint-probability 0.1 \
  --loss-weight-power 1.0 \
  --eval-every 100 --sample-every 1000 --checkpoint-every 2000 \
  --num-eval-noises 16 --num-workers 4 --output-dir outputs/imagenette10

## Visualize progress

Colab runs one cell at a time, so to check on a long run: interrupt the training cell (progress up to the last checkpoint is safe), run this, then continue with the resume cell at the bottom. Shows real training images vs the latest one-step samples, plus the loss curves.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

out = Path("outputs/imagenette10")
# Sort by step number: filenames are zero-padded to 4 digits, so past step 9999
# alphabetical order puts e.g. step 9000 after step 20000.
sample_grids = sorted(out.glob("sample_step_*.png"), key=lambda p: int(p.stem.split("_")[-1]))

fig, axes = plt.subplots(2, 1, figsize=(16, 5))
axes[0].imshow(Image.open(out / "train_batch_grid.png"))
axes[0].set_title("training data (reference)")
axes[0].axis("off")
axes[1].imshow(Image.open(sample_grids[-1]))
axes[1].set_title(f"one-step samples ({sample_grids[-1].name})")
axes[1].axis("off")
plt.tight_layout()
plt.show()

history = pd.read_csv(out / "loss_history.csv")
plt.figure(figsize=(7, 4))
plt.semilogy(history["step"], history["loss"], alpha=0.5, label="loss")
plt.semilogy(history["step"], history["smoothed_loss"], "k", linewidth=2, label="smoothed_loss")
plt.xlabel("step")
plt.legend()
plt.title("Training curves")
plt.show()

## Save outputs to Google Drive

Everything in `outputs/` is lost when the runtime ends. Run this after the smoke test, and **periodically during/after the main run** (interrupt the training cell, save, then resume via the cell below if needed).

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import shutil
from pathlib import Path

drive_dir = Path('/content/drive/MyDrive/MeanFlow_outputs')
drive_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree('outputs', drive_dir, dirs_exist_ok=True)
print(f"Copied outputs/ to {drive_dir}")

## Resume after a disconnect

Rerun the **setup** and **dataset** cells first, then this cell to restore the checkpoint from Drive, then the training cell below it.

Notes:
- `--resume-from` restores model weights (EMA-smoothed), the EMA state, and the Adam optimizer state.
- The cosine schedule does not resume mid-curve: pass `--no-lr-decay` with `--lr` set near where the previous leg left off (the `lr=` value printed in its last log lines), or a fresh shorter cosine via `--lr-decay-steps`.
- Each leg writes a fresh `loss_history.csv` (by design); keep the Drive copies if you want the full curve across legs.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

!mkdir -p outputs/imagenette10
!cp /content/drive/MyDrive/MeanFlow_outputs/imagenette10/checkpoint.pt outputs/imagenette10/checkpoint_resume.pt

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train.py --data-dir data/imagenette2-160/train \
  --resume-from outputs/imagenette10/checkpoint_resume.pt \
  --steps 30000 --batch-size 64 --no-lr-decay --lr 1e-4 \
  --time-sampling logit_normal --equal-time-probability 0.5 --endpoint-probability 0.1 \
  --loss-weight-power 1.0 \
  --eval-every 100 --sample-every 1000 --checkpoint-every 2000 \
  --num-eval-noises 16 --num-workers 4 --output-dir outputs/imagenette10